# 🖼️ Image Classification: CNN vs. ViT

**CO5085 – Assignment 1**

So sánh: **ResNet-50, EfficientNet-B0** (CNN) vs. **ViT-B/16** (Vision Transformer)

In [ ]:
import sys, torch, os, json
sys.path.insert(0, '../src')
os.makedirs('../results', exist_ok=True)
DEVICE = 'cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu'

TRAIN_MODE = False  # False = load checkpoint (nhanh) | True = train lại từ đầu
print(f'Device: {DEVICE} | Train mode: {TRAIN_MODE}')

## 1. Load Data

In [ ]:
from datasets import get_cifar100_loaders
train_loader, val_loader, test_loader = get_cifar100_loaders(
    data_dir='../data/image', batch_size=128)
print(f'Train batches: {len(train_loader)} | Val: {len(val_loader)} | Test: {len(test_loader)}')

## 2. Model 1: ResNet-50 (CNN)

In [ ]:
from models import get_resnet50

resnet = get_resnet50(num_classes=100, pretrained=not TRAIN_MODE)
if TRAIN_MODE:
    from train import train
    history_resnet = train(resnet, train_loader, val_loader,
                           num_epochs=5, lr=1e-3, device=DEVICE,
                           save_path='../results/resnet50_best.pt', scheduler_type='cosine')
else:
    resnet.load_state_dict(torch.load('../results/resnet50_best.pt', map_location=DEVICE))
    with open('../results/resnet50_history.json') as f: history_resnet = json.load(f)
    print('ResNet-50 loaded ✅')
resnet = resnet.to(DEVICE).eval()

## 3. Model 2: EfficientNet-B0 (CNN)

In [ ]:
from models import get_efficientnet_b0

effnet = get_efficientnet_b0(num_classes=100, pretrained=not TRAIN_MODE)
if TRAIN_MODE:
    from train import train
    history_effnet = train(effnet, train_loader, val_loader,
                           num_epochs=5, lr=1e-3, device=DEVICE,
                           save_path='../results/efficientnet_best.pt')
else:
    effnet.load_state_dict(torch.load('../results/efficientnet_best.pt', map_location=DEVICE))
    with open('../results/efficientnet_history.json') as f: history_effnet = json.load(f)
    print('EfficientNet-B0 loaded ✅')
effnet = effnet.to(DEVICE).eval()

## 4. Model 3: ViT-B/16

In [ ]:
from models import get_vit_b16
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

# ViT needs 224x224
normalize = transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
val_tf = transforms.Compose([transforms.Resize(256), transforms.CenterCrop(224), transforms.ToTensor(), normalize])
test_ds_vit = datasets.CIFAR100('../data/image', train=False, download=False, transform=val_tf)
test_loader_vit = DataLoader(test_ds_vit, batch_size=32, shuffle=False, num_workers=0)

vit_model = get_vit_b16(num_classes=100, pretrained=not TRAIN_MODE)
if TRAIN_MODE:
    from train import train
    train_tf = transforms.Compose([transforms.Resize(256), transforms.RandomCrop(224),
                                    transforms.RandomHorizontalFlip(), transforms.ToTensor(), normalize])
    train_ds_vit = datasets.CIFAR100('../data/image', train=True, download=False, transform=train_tf)
    val_size = int(0.2 * len(train_ds_vit))
    train_ds_vit, val_ds_vit = torch.utils.data.random_split(
        train_ds_vit, [len(train_ds_vit)-val_size, val_size],
        generator=torch.Generator().manual_seed(42))
    train_loader_vit = DataLoader(train_ds_vit, batch_size=32, shuffle=True, num_workers=0)
    val_loader_vit   = DataLoader(val_ds_vit,   batch_size=32, shuffle=False, num_workers=0)
    history_vit = train(vit_model, train_loader_vit, val_loader_vit,
                        num_epochs=5, lr=2e-5, device=DEVICE,
                        save_path='../results/vit_b16_best.pt', scheduler_type='cosine')
else:
    vit_model.load_state_dict(torch.load('../results/vit_b16_best.pt', map_location=DEVICE))
    with open('../results/vit_b16_history.json') as f: history_vit = json.load(f)
    print('ViT-B/16 loaded ✅')
vit_model = vit_model.to(DEVICE).eval()

## 5. Evaluation & Comparison

In [ ]:
from evaluate import get_predictions, compute_metrics, plot_training_curves, compare_models
import torch

results = {}
for model_name, model, hist, loader in [
    ('ResNet-50',       resnet,      history_resnet, test_loader),
    ('EfficientNet-B0', effnet,      history_effnet, test_loader),
    ('ViT-B/16',        vit_model,   history_vit,    test_loader_vit),
]:
    print(f'\n=== {model_name} ===')
    preds, labels, probs = get_predictions(model, loader, DEVICE)
    results[model_name] = compute_metrics(preds, labels, verbose=True)
    plot_training_curves(hist, model_name=model_name, save_path=f'../results/{model_name}_curves.png')

## 6. Overall Comparison Chart

In [ ]:
from evaluate import compare_models
compare_models(results, metric='accuracy', save_path='../results/image_comparison_acc.png')
compare_models(results, metric='f1_macro', save_path='../results/image_comparison_f1.png')